Zaimplementuj aplikację szacującą czas ukończenia półmaratonu dla zadanych danych

1. Umieść dane w Digital Ocean Spaces

1. Napisz notebook, który będzie Twoim pipelinem do trenowania modelu
    * czyta dane z Digital Ocean Spaces
    * czyści je
    * trenuje model (dobierz odpowiednie metryki [feature selection])
    * nowa wersja modelu jest zapisywana lokalnie i do Digital Ocean Spaces

1. Aplikacja
    * opakuj model w aplikację streamlit
    * wdróż (deploy) aplikację za pomocą Digital Ocean AppPlatform 
    * wejściem jest pole tekstowe, w którym użytkownik się przedstawia, mówi o tym
    jaka jest jego płeć, wiek i czas na 5km
    * jeśli użytkownik podał za mało danych, wyświetl informację o tym jakich danych brakuje
    * za pomocą LLM (OpenAI) wyłuskaj potrzebne dane, potrzebne dla Twojego modelu
    do określenia, do słownika (dictionary lub JSON)
    * tę część podepnij do Langfuse, aby zbierać metryki o skuteczności działania LLM'a



# Wczytanie danych i wstępna inspekcja

In [78]:
import pandas as pd
import numpy as np
import pathlib as Path
from pathlib import Path
from IPython.display import display
from dotenv import load_dotenv
import os

load_dotenv()

True

In [79]:

print(os.getenv("AWS_ACCESS_KEY_ID"))

DO00A4DWQ6MHNQE9R9J6


In [80]:
import boto3

s3 = boto3.client(
    "s3",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    endpoint_url=os.getenv("AWS_ENDPOINT_URL_S3"),
)

In [81]:
response = s3.list_buckets()

for bucket in response["Buckets"]:
    print(bucket["Name"])

halfmarathon-data


In [82]:
DATA_DIR = Path("data")
file_path = DATA_DIR / "halfmarathon_wroclaw_2023__final.csv"
BUCKET_NAME = "halfmarathon-data"

s3.upload_file(
    Filename=str(file_path),
    Bucket=BUCKET_NAME,
    Key=file_path.name

)

print("Plik został wysłany.")

Plik został wysłany.


In [83]:
DATA_DIR = Path("data")

for file in DATA_DIR.glob("*.csv"):
    print(f"Wysyłam: {file.name}")

    s3.upload_file(
        Filename=str(file),
        Bucket=BUCKET_NAME,
        Key=file.name
    )

print("Wszystkie pliki zostały wysłane.")

Wysyłam: halfmarathon_wroclaw_2023__final.csv
Wysyłam: halfmarathon_wroclaw_2024__final.csv
Wszystkie pliki zostały wysłane.


In [84]:
response = s3.list_objects_v2(Bucket=BUCKET_NAME)

for obj in response.get("Contents", []):
    print(obj["Key"])

halfmarathon_wroclaw_2023__final.csv
halfmarathon_wroclaw_2024__final.csv
models/halfmarathon_linear_regression.pkl


In [85]:
response = s3.get_object(
    Bucket=BUCKET_NAME,
    Key="halfmarathon_wroclaw_2023__final.csv"
)

df_2023 = pd.read_csv(response["Body"], sep=";")

In [86]:
response = s3.get_object(
    Bucket=BUCKET_NAME,
    Key="halfmarathon_wroclaw_2024__final.csv"
)

df_2024 = pd.read_csv(response["Body"], sep=";")

In [87]:
print("2023")
display(df_2023.head())

print("2024")
display(df_2024.head())

2023


,Miejsce,Numer startowy,Imię,Nazwisko,Miasto,Kraj,Drużyna,Płeć,Płeć Miejsce,Kategoria wiekowa,...,10 km Tempo,15 km Czas,15 km Miejsce Open,15 km Tempo,20 km Czas,20 km Miejsce Open,20 km Tempo,Tempo Stabilność,Czas,Tempo
0,1.0,1787,TOMASZ,GRYCKO,NaN,POL,UKS BLIZA WŁADYSŁAWOWO,M,1.0,M30,...,2.926667,00:44:47,1.0,3.106667,01:01:43,1.0,3.386667,0.031400,01:04:59,3.080509
1,2.0,3,ARKADIUSZ,GARDZIELEWSKI,WROCŁAW,POL,ARKADIUSZGARDZIELEWSKI.PL,M,2.0,M30,...,2.983333,00:45:26,2.0,3.143333,01:03:08,2.0,3.540000,0.038000,01:06:23,3.146875
2,3.0,3832,KRZYSZTOF,HADAS,POZNAŃ,POL,NaN,M,3.0,M20,...,3.123333,00:47:34,3.0,3.236667,01:05:09,3.0,3.516667,0.024067,01:08:24,3.242475
3,4.0,416,DAMIAN,DYDUCH,KĘPNO,POL,AZS POLITECHNIKA OPOLSKA,M,4.0,M30,...,3.196667,00:48:49,5.0,3.330000,01:06:54,4.0,3.616667,0.025467,01:10:16,3.330963
4,5.0,8476,KAMIL,MAŃKOWSKI,MIRKÓW,POL,PARKRUN WROCŁAW,M,5.0,M20,...,3.276667,00:49:31,7.0,3.386667,01:07:27,5.0,3.586667,0.023000,01:10:27,3.339654


2024


,Miejsce,Numer startowy,Imię,Nazwisko,Miasto,Kraj,Drużyna,Płeć,Płeć Miejsce,Kategoria wiekowa,...,10 km Tempo,15 km Czas,15 km Miejsce Open,15 km Tempo,20 km Czas,20 km Miejsce Open,20 km Tempo,Tempo Stabilność,Czas,Tempo
0,1.0,596,NIKODEM,DWORCZAK,KOŚCIAN,POL,NaN,M,1.0,M20,...,2.920000,00:45:07,2.0,3.083333,01:00:33,1.0,3.086667,0.007267,01:04:03,3.036265
1,2.0,616,MATEUSZ,KACZOR,RADOM,POL,RLTL OPTIMA RADOM,M,2.0,M20,...,2.920000,00:45:07,3.0,3.083333,01:00:38,2.0,3.103333,0.008267,01:04:24,3.052856
2,3.0,154,PATRYK,KOZŁOWSKI,RADOM,POL,RLTL-ZTE-RADOM,M,3.0,M20,...,2.920000,00:45:07,1.0,3.083333,01:00:59,3.0,3.173333,0.012467,01:04:40,3.065497
3,4.0,591,DARIUSZ,BORATYŃSKI,WROCŁAW,POL,WOSIEK TEAM AZS AWF WROCŁAW,M,4.0,M20,...,3.110000,00:47:48,4.0,3.293333,01:05:40,4.0,3.573333,0.028667,01:09:44,3.305681
4,5.0,521,SZYMON,DOROŻYŃSKI,LUBON,POL,SZYMI TEAM AZS POLITECHNIKA OPOLSKA,M,5.0,M30,...,3.153333,00:48:09,5.0,3.453333,01:06:05,5.0,3.586667,0.039800,01:10:05,3.322272


In [88]:
print("Kolumny 2023:")
for col in df_2023.columns:
    print(col)

print("\n" + "="*50 + "\n")

print("Kolumny 2024:")
for col in df_2024.columns:
    print(col)

Kolumny 2023:
Miejsce
Numer startowy
Imię
Nazwisko
Miasto
Kraj
Drużyna
Płeć
Płeć Miejsce
Kategoria wiekowa
Kategoria wiekowa Miejsce
Rocznik
5 km Czas
5 km Miejsce Open
5 km Tempo
10 km Czas
10 km Miejsce Open
10 km Tempo
15 km Czas
15 km Miejsce Open
15 km Tempo
20 km Czas
20 km Miejsce Open
20 km Tempo
Tempo Stabilność
Czas
Tempo


Kolumny 2024:
Miejsce
Numer startowy
Imię
Nazwisko
Miasto
Kraj
Drużyna
Płeć
Płeć Miejsce
Kategoria wiekowa
Kategoria wiekowa Miejsce
Rocznik
5 km Czas
5 km Miejsce Open
5 km Tempo
10 km Czas
10 km Miejsce Open
10 km Tempo
15 km Czas
15 km Miejsce Open
15 km Tempo
20 km Czas
20 km Miejsce Open
20 km Tempo
Tempo Stabilność
Czas
Tempo


In [89]:
cols_2023 = set(df_2023.columns)
cols_2024 = set(df_2024.columns)

print("Tylko w 2023:")
print(sorted(cols_2023 - cols_2024))
      
print("\nTylko w 2024:")
print(sorted(cols_2024 - cols_2023))

print("\nCzy kolumny są identyczne?")
print(cols_2023 == cols_2024)

Tylko w 2023:
[]

Tylko w 2024:
[]

Czy kolumny są identyczne?
True


In [90]:
time_cols_2023 = [col for col in df_2023.columns if "czas" in col.lower() or "km" in col.lower()]
time_cols_2024 = [col for col in df_2024.columns if "czas" in col.lower() or "km" in col.lower()]

print("Kolumny czasowe 2023:")
print(time_cols_2023)

print("\nKolumny czasowe 2024:")
print(time_cols_2024)

Kolumny czasowe 2023:
['5 km Czas', '5 km Miejsce Open', '5 km Tempo', '10 km Czas', '10 km Miejsce Open', '10 km Tempo', '15 km Czas', '15 km Miejsce Open', '15 km Tempo', '20 km Czas', '20 km Miejsce Open', '20 km Tempo', 'Czas']

Kolumny czasowe 2024:
['5 km Czas', '5 km Miejsce Open', '5 km Tempo', '10 km Czas', '10 km Miejsce Open', '10 km Tempo', '15 km Czas', '15 km Miejsce Open', '15 km Tempo', '20 km Czas', '20 km Miejsce Open', '20 km Tempo', 'Czas']


In [91]:
for col in df_2024.columns:
    if "czas" in col.lower():
        print(col)

5 km Czas
10 km Czas
15 km Czas
20 km Czas
Czas


In [92]:
for col in df_2023.columns:
    if "czas" in col.lower():
        print(col)

5 km Czas
10 km Czas
15 km Czas
20 km Czas
Czas


In [93]:
df_2023 = df_2023.copy()
df_2024 = df_2024.copy()

df_2023["Rok zawodów"] = 2023
df_2024["Rok zawodów"] = 2024

df = pd.concat([df_2023, df_2024], ignore_index=True)

print("Połączony dataset:", df.shape)
display(df.head())

Połączony dataset: (21957, 28)


,Miejsce,Numer startowy,Imię,Nazwisko,Miasto,Kraj,Drużyna,Płeć,Płeć Miejsce,Kategoria wiekowa,...,15 km Czas,15 km Miejsce Open,15 km Tempo,20 km Czas,20 km Miejsce Open,20 km Tempo,Tempo Stabilność,Czas,Tempo,Rok zawodów
0,1.0,1787,TOMASZ,GRYCKO,NaN,POL,UKS BLIZA WŁADYSŁAWOWO,M,1.0,M30,...,00:44:47,1.0,3.106667,01:01:43,1.0,3.386667,0.031400,01:04:59,3.080509,2023
1,2.0,3,ARKADIUSZ,GARDZIELEWSKI,WROCŁAW,POL,ARKADIUSZGARDZIELEWSKI.PL,M,2.0,M30,...,00:45:26,2.0,3.143333,01:03:08,2.0,3.540000,0.038000,01:06:23,3.146875,2023
2,3.0,3832,KRZYSZTOF,HADAS,POZNAŃ,POL,NaN,M,3.0,M20,...,00:47:34,3.0,3.236667,01:05:09,3.0,3.516667,0.024067,01:08:24,3.242475,2023
3,4.0,416,DAMIAN,DYDUCH,KĘPNO,POL,AZS POLITECHNIKA OPOLSKA,M,4.0,M30,...,00:48:49,5.0,3.330000,01:06:54,4.0,3.616667,0.025467,01:10:16,3.330963,2023
4,5.0,8476,KAMIL,MAŃKOWSKI,MIRKÓW,POL,PARKRUN WROCŁAW,M,5.0,M20,...,00:49:31,7.0,3.386667,01:07:27,5.0,3.586667,0.023000,01:10:27,3.339654,2023


In [94]:
model_cols = ["Płeć", "Rocznik", "5 km Czas", "Czas", "Rok zawodów"]

model_df = df[model_cols].copy()

print(model_df.shape)
print(model_df.head())

(21957, 5)
  Płeć  Rocznik 5 km Czas      Czas  Rok zawodów
0    M   1992.0  00:14:37  01:04:59         2023
1    M   1986.0  00:14:48  01:06:23         2023
2    M   1996.0  00:15:46  01:08:24         2023
3    M   1988.0  00:16:11  01:10:16         2023
4    M   1995.0  00:16:12  01:10:27         2023


In [95]:
missing_summary = (
    model_df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_summary["missing_pct"] = (
    missing_summary["missing_count"] / len(model_df) * 100
).round(2)

display(missing_summary)

,missing_count,missing_pct
Płeć,11,0.05
Rocznik,485,2.21
5 km Czas,3546,16.15
Czas,2055,9.36
Rok zawodów,0,0.00


In [96]:
model_df["Wiek"] = model_df["Rok zawodów"] - model_df["Rocznik"]
display(model_df.head())

,Płeć,Rocznik,5 km Czas,Czas,Rok zawodów,Wiek
0,M,1992.0,00:14:37,01:04:59,2023,31.0
1,M,1986.0,00:14:48,01:06:23,2023,37.0
2,M,1996.0,00:15:46,01:08:24,2023,27.0
3,M,1988.0,00:16:11,01:10:16,2023,35.0
4,M,1995.0,00:16:12,01:10:27,2023,28.0


In [97]:
model_df = model_df[["Płeć", "Wiek", "5 km Czas", "Czas", "Rocznik", "Rok zawodów"]]

display(model_df.head())

,Płeć,Wiek,5 km Czas,Czas,Rocznik,Rok zawodów
0,M,31.0,00:14:37,01:04:59,1992.0,2023
1,M,37.0,00:14:48,01:06:23,1986.0,2023
2,M,27.0,00:15:46,01:08:24,1996.0,2023
3,M,35.0,00:16:11,01:10:16,1988.0,2023
4,M,28.0,00:16:12,01:10:27,1995.0,2023


In [98]:
print("Przykładowe wartości 5 km Czas:")
print(model_df["5 km Czas"].head(10).tolist())

print("\nPrzykładowe wartości Czas:")
print(model_df["Czas"].head(10).tolist())

Przykładowe wartości 5 km Czas:
['00:14:37', '00:14:48', '00:15:46', '00:16:11', '00:16:12', '00:16:09', '00:15:37', '00:16:30', '00:17:10', '00:16:53']

Przykładowe wartości Czas:
['01:04:59', '01:06:23', '01:08:24', '01:10:16', '01:10:27', '01:10:34', '01:11:18', '01:11:42', '01:14:16', '01:14:22']


In [99]:
def time_to_seconds(time_str):
    """
    Zamienia czas w formacie HH:MM:SS na liczbe sekund.
    Jeśli wartość jest pusta lub błędna, zwraca np.nan.
    """
    if pd.isna(time_str):
        return np.nan

    time_str = str(time_str).strip()

    try:
        parts = time_str.split(":")
        if len(parts) !=3:
            return np.nan
        
        hours, minutes, seconds = map(int, parts)
        return hours * 3600 + minutes * 60 + seconds

    except Exception:
        return np.nan

In [100]:
model_df["5 km Czas sek"] = model_df["5 km Czas"].apply(time_to_seconds)
model_df["Czas sek"] = model_df["Czas"].apply(time_to_seconds)

display(model_df.head())

,Płeć,Wiek,5 km Czas,Czas,Rocznik,Rok zawodów,5 km Czas sek,Czas sek
0,M,31.0,00:14:37,01:04:59,1992.0,2023,877.0,3899.0
1,M,37.0,00:14:48,01:06:23,1986.0,2023,888.0,3983.0
2,M,27.0,00:15:46,01:08:24,1996.0,2023,946.0,4104.0
3,M,35.0,00:16:11,01:10:16,1988.0,2023,971.0,4216.0
4,M,28.0,00:16:12,01:10:27,1995.0,2023,972.0,4227.0


In [101]:
display(
    model_df[
        ["5 km Czas", "5 km Czas sek", "Czas", "Czas sek"]
        ].head(10))

,5 km Czas,5 km Czas sek,Czas,Czas sek
0,00:14:37,877.0,01:04:59,3899.0
1,00:14:48,888.0,01:06:23,3983.0
2,00:15:46,946.0,01:08:24,4104.0
3,00:16:11,971.0,01:10:16,4216.0
4,00:16:12,972.0,01:10:27,4227.0
5,00:16:09,969.0,01:10:34,4234.0
6,00:15:37,937.0,01:11:18,4278.0
7,00:16:30,990.0,01:11:42,4302.0
8,00:17:10,1030.0,01:14:16,4456.0
9,00:16:53,1013.0,01:14:22,4462.0


In [102]:
missing_after_conversion = (
    model_df[["Płeć", "Wiek", "5 km Czas sek", "Czas sek"]]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_after_conversion["missing_pct"] = (
    missing_after_conversion["missing_count"] / len(model_df) * 100
).round(2)

display(missing_after_conversion)

,missing_count,missing_pct
Płeć,11,0.05
Wiek,485,2.21
5 km Czas sek,3546,16.15
Czas sek,3507,15.97


In [103]:
clean_df = model_df.dropna(subset=["Płeć", "Wiek", "5 km Czas sek", "Czas sek"]).copy()

print("Liczba rekordów przed czyszczeniem:", len(model_df))
print("Liczba rekordów po czyszczeniu:", len(clean_df))

display(clean_df.head())

Liczba rekordów przed czyszczeniem: 21957
Liczba rekordów po czyszczeniu: 17927


,Płeć,Wiek,5 km Czas,Czas,Rocznik,Rok zawodów,5 km Czas sek,Czas sek
0,M,31.0,00:14:37,01:04:59,1992.0,2023,877.0,3899.0
1,M,37.0,00:14:48,01:06:23,1986.0,2023,888.0,3983.0
2,M,27.0,00:15:46,01:08:24,1996.0,2023,946.0,4104.0
3,M,35.0,00:16:11,01:10:16,1988.0,2023,971.0,4216.0
4,M,28.0,00:16:12,01:10:27,1995.0,2023,972.0,4227.0


In [104]:
final_df = clean_df[["Płeć", "Wiek", "5 km Czas sek", "Czas sek"]].copy()

display(final_df.head())
print(final_df.shape)

,Płeć,Wiek,5 km Czas sek,Czas sek
0,M,31.0,877.0,3899.0
1,M,37.0,888.0,3983.0
2,M,27.0,946.0,4104.0
3,M,35.0,971.0,4216.0
4,M,28.0,972.0,4227.0


(17927, 4)


# Przygotowanie danych do notebooka 

In [105]:
final_df = final_df.copy()

X = final_df[["Płeć", "Wiek", "5 km Czas sek"]]
y = final_df["Czas sek"]

display(X.head())
display(y.head())
print(X.shape, y.shape)

,Płeć,Wiek,5 km Czas sek
0,M,31.0,877.0
1,M,37.0,888.0
2,M,27.0,946.0
3,M,35.0,971.0
4,M,28.0,972.0


0    3899.0
1    3983.0
2    4104.0
3    4216.0
4    4227.0
Name: Czas sek, dtype: float64

(17927, 3) (17927,)


In [106]:
X = X.copy()

X["Płeć"] = X["Płeć"].map({"M": 1, "K": 0})

display(X.head(10))
print(X["Płeć"].value_counts(dropna=False))
print(X.dtypes)

,Płeć,Wiek,5 km Czas sek
0,1,31.0,877.0
1,1,37.0,888.0
2,1,27.0,946.0
3,1,35.0,971.0
4,1,28.0,972.0
5,1,40.0,969.0
6,1,24.0,937.0
7,1,34.0,990.0
8,1,22.0,1030.0
9,1,31.0,1013.0


Płeć
1    12760
0     5167
Name: count, dtype: int64
Płeć               int64
Wiek             float64
5 km Czas sek    float64
dtype: object


In [107]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (14341, 3)
X_test: (3586, 3)
y_train: (14341,)
y_test: (3586,)


In [108]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](3,)","[ 7.88,-0. , 4.59]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](3,)","['Płeć','Wiek','5 km Czas sek']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-296.9
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,3
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,3


In [109]:
y_pred = lin_reg.predict(X_test)

print(y_pred[:10])

[ 7176.08257961  8817.244118    8062.62723394  7309.08266019
 10158.84189847  7117.64443744  7797.50215419  9041.07501276
  8633.68675312  6257.1875166 ]


In [110]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f} sek")
print(f"RMSE: {rmse:.2f} sek")
print(f"R²: {r2:.4f}")

MAE: 301.53 sek
RMSE: 423.15 sek
R²: 0.8813


### Random Forest Regressor

In [111]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"m

In [112]:
y_pred_rf = rf.predict(X_test)

print(y_pred_rf[:10])

[7135.995      9004.795      7971.09666667 7132.895      9969.155
 7165.95458333 7652.90133333 9073.961      8845.965      6315.83983333]


In [113]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print(f"RF MAE: {mae_rf:.2f} sek")
print(f"RF RMSE: {rmse_rf:.2f} sek")
print(f"RF R²: {r2_rf:.4f}")


RF MAE: 335.73 sek
RF RMSE: 459.52 sek
RF R²: 0.8600


In [114]:
comparison_df = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [mae, mae_rf],
    "RMSE": [rmse, rmse_rf],
    "R2": [r2, r2_rf] 
})

display(comparison_df)

,Model,MAE,RMSE,R2
0,Linear Regression,301.530991,423.150075,0.881283
1,Random Forest,335.731322,459.517017,0.860000


 ### Zapisanie modelu finalnego


In [115]:
import joblib
from pathlib import Path

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

model_path = MODELS_DIR / "halfmarathon_linear_regression.pkl"

joblib.dump(lin_reg, model_path)

print(f"Model zapisany do: {model_path}")
 





Model zapisany do: models\halfmarathon_linear_regression.pkl


In [116]:
def seconds_to_hhmmss(total_seconds):
    total_seconds = int(round(total_seconds))

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

In [117]:
print(seconds_to_hhmmss(6746))

01:52:26


In [118]:
def predict_halfmarathon_time(model, plec, wiek, czas_5km_sek):
    """
    Przewiduje czas ukończenia półmaratonu
    """

    plec_map = {
        "M": 1,
        "K": 0,
    }

    input_df = pd.DataFrame({
        "Płeć": [plec_map[plec]],
        "Wiek": [wiek],
        "5 km Czas sek": [czas_5km_sek]
    })

    predicted_seconds = model.predict(input_df)[0]

    predicted_time = seconds_to_hhmmss(predicted_seconds)

    return predicted_seconds, predicted_time

In [119]:
pred_seconds, pred_time = predict_halfmarathon_time(
    model=lin_reg,
    plec="M",
    wiek=30,
    czas_5km_sek=1500
)

print("Sekundy:", round(pred_seconds))
print("Czas:", pred_time)

Sekundy: 6602
Czas: 01:50:02


In [120]:
s3.upload_file(
    Filename="models/halfmarathon_linear_regression.pkl",
    Bucket=BUCKET_NAME,
    Key="models/halfmarathon_linear_regression.pkl"
)

In [121]:
import os

download_path = "models/downloaded_halfmarathon_linear_regression.pkl"

s3.download_file(
    Bucket=BUCKET_NAME,
    Key="models/halfmarathon_linear_regression.pkl",
    Filename=download_path
)

print("Model pobrany.")
print(os.path.exists(download_path))

Model pobrany.
True


In [122]:
import joblib

loaded_model = joblib.load(download_path)

print(type(loaded_model))

<class 'sklearn.linear_model._base.LinearRegression'>
